# Audio Deepfake Detection v1

Baseline notebook for short speech deepfake detection.

Planned setup:
- dataset via CSV manifest
- mono 16 kHz audio
- fixed-duration clips
- log-Mel spectrogram input
- lightweight CNN classifier


In [ ]:
# Install dependencies in Colab if needed
# !pip install torchaudio soundfile


In [ ]:
import os
import csv
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
MANIFEST_PATH = 'audio/data/manifests/asvspoof_df_manifest.csv'
SAMPLE_RATE = 16000
CLIP_SECONDS = 4
NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS
N_MELS = 80
BATCH_SIZE = 32
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
PIN_MEMORY = torch.cuda.is_available()
LEARNING_RATE = 1e-3
EPOCHS = 10

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f'Manifest not found: {MANIFEST_PATH}')

print(f'Manifest: {MANIFEST_PATH}')


In [ ]:
def load_manifest(path):
    rows = []
    with open(path, 'r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            rows.append({
                'audio_path': row['audio_path'],
                'split': row['split'],
                'label': 0 if row['label'].strip().lower() == 'real' else 1,
                'source': row.get('source', ''),
            })
    return rows

rows = load_manifest(MANIFEST_PATH)
train_rows = [row for row in rows if row['split'] == 'train']
val_rows = [row for row in rows if row['split'] == 'val']
test_rows = [row for row in rows if row['split'] == 'test']

print(f'train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}')


In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    n_mels=N_MELS,
)
amplitude_to_db = T.AmplitudeToDB()

def load_waveform(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    if waveform.size(0) > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != SAMPLE_RATE:
        waveform = T.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)(waveform)
    return waveform

def pad_or_crop(waveform):
    length = waveform.size(-1)
    if length > NUM_SAMPLES:
        start = random.randint(0, length - NUM_SAMPLES)
        waveform = waveform[:, start:start + NUM_SAMPLES]
    elif length < NUM_SAMPLES:
        pad_amount = NUM_SAMPLES - length
        waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
    return waveform

class AudioDeepfakeDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        label = row['label']
        audio_path = row['audio_path']

        try:
            waveform = load_waveform(audio_path)
            waveform = pad_or_crop(waveform)
            features = mel_transform(waveform)
            features = amplitude_to_db(features)
            features = (features - features.mean()) / (features.std() + 1e-6)
            return features, torch.tensor(label, dtype=torch.float32)
        except Exception:
            new_idx = (idx + 1) % len(self.rows)
            return self.__getitem__(new_idx)


In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, 1)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x).squeeze(-1)

model = AudioCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model


In [ ]:
train_dataset = AudioDeepfakeDataset(train_rows)
val_dataset = AudioDeepfakeDataset(val_rows)
test_dataset = AudioDeepfakeDataset(test_rows)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f'train batches={len(train_loader)} val batches={len(val_loader)} test batches={len(test_loader)}')
batch_x, batch_y = next(iter(train_loader))
print('batch_x', batch_x.shape, 'batch_y', batch_y.shape)

# TODO next:
# - implement train loop
# - compute AUC / EER / F1
# - save checkpoint as audio_deepfake_v1.pth
